# Clustering & PCA

> 📘 **Python Mastery** · Module 13 — Machine Learning · Lesson 6/7

No labels, no answer key — yet structure hides in the data. Clustering finds the
groups; PCA finds the shadow that keeps the shape.

## 🎯 Learning Objectives

- Explain what changes when we drop labels from the problem entirely.
- Generate clustered synthetic data with `make_blobs`.
- Run K-Means and inspect `labels_` and `cluster_centers_`.
- Choose the number of clusters with the elbow method over `inertia_`.
- State K-Means' assumptions and recognise when they break.
- Compress iris from 4 features to 2 with PCA and read `explained_variance_ratio_`.
- Justify scaling before PCA, and know why pipelines often end in fewer dimensions.

## 1. The Unsupervised Setup

Until now every model saw questions *with* answers. Unsupervised learning gets only
the questions. Two classic jobs:

| Task | Question | Example |
|---|---|---|
| Clustering | "What natural groups exist?" | Segment telecom customers by usage |
| Dimensionality reduction | "Which few directions carry most of the information?" | Squeeze 64 pixels into 2 plotting axes |

There's no accuracy score here — evaluation becomes about interpretability,
stability, and downstream usefulness.

**Syntax:**
```python
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)
```

> 🔍 **Under the Hood:** K-Means runs **Lloyd's algorithm** — drop k random points as
> seed centroids (why you need `random_state`!), assign every point to its nearest
> centroid, recompute each centroid as the mean of its members, repeat until nothing
> moves. It converges to a *local* optimum, so different seeds can give different
> groupings; `n_init` reruns the whole dance several times and keeps the lowest-energy
> result. `inertia_` is the final energy: the sum of squared distances from each point
> to its centroid.

## 2. Manufacturing Clustered Data

We invent a mall-customers dataset with three hidden tribes (budget, regular, VIP)
— then pretend we don't know the tribes exist.

In [ ]:
# Three latent customer groups, two spending features
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
centers_true = np.array([[20, 20], [55, 60], [85, 30]])   # (monthly_visits, avg_spend)
spreads      = np.array([6.0, 7.0, 5.0])

parts = [rng.normal(loc=c, scale=s, size=(80, 2))
         for c, s in zip(centers_true, spreads)]
X = np.vstack(parts)

plt.figure(figsize=(6.5, 4))
plt.scatter(X[:, 0], X[:, 1], s=24, alpha=0.75, color="#1f77b4")
plt.title("Mall customers - groups not visible yet")
plt.xlabel("Monthly visits"); plt.ylabel("Avg spend ('000 BDT)")
plt.grid(alpha=0.3)
plt.show()

## 3. K-Means: Find the Groups

Fit, then read two artefacts: `labels_` (group number per customer) and
`cluster_centers_` (each group's centre of gravity).

In [ ]:
# Ask for 3 groups and colour them in
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
labels = kmeans.labels_
centers = kmeans.cluster_centers_

colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
plt.figure(figsize=(6.5, 4))
for k in range(3):
    pts = X[labels == k]
    plt.scatter(pts[:, 0], pts[:, 1], s=24, alpha=0.75,
                color=colors[k], label=f"cluster {k}")
plt.scatter(centers[:, 0], centers[:, 1], marker="X", s=220,
            color="#d62728", edgecolor="white", linewidth=1.5,
            label="centroids", zorder=3)
plt.title("K-Means recovered the three tribes")
plt.xlabel("Monthly visits"); plt.ylabel("Avg spend ('000 BDT)")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print("cluster sizes:", np.bincount(labels).tolist())
print("cluster centers:\n", centers.round(1))

In [ ]:
# Inference: which tribe does a NEW customer belong to?
import pandas as pd

newcomers = pd.DataFrame({"monthly_visits": [25, 70],
                          "avg_spend_k": [22, 65]})
newcomers["tribe"] = kmeans.predict(newcomers.to_numpy())
print(newcomers.to_string(index=False))

## 4. Choosing k: The Elbow Method

In real life nobody tells you there were three tribes. Sweep k from 1 upward and plot
`inertia_`. It always falls as k grows — but after the true group count, extra
clusters stop buying much. That bend is the **elbow**, and it's your k.

In [ ]:
# Inertia across k = 1..8, bend annotated
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

ks = range(1, 9)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_
            for k in ks]

plt.figure(figsize=(6.5, 4))
plt.plot(list(ks), inertias, marker="o", color="#1f77b4", linewidth=2)
plt.annotate("elbow: extra clusters stop paying off",
             xy=(3, inertias[2]), xytext=(4.3, inertias[2] * 1.35),
             arrowprops=dict(arrowstyle="->", color="#d62728"), color="#d62728")
plt.title("Elbow method: pick k where the fall flattens")
plt.xlabel("number of clusters k"); plt.ylabel("inertia (within-cluster SSE)")
plt.grid(alpha=0.3)
plt.show()

for k, ine in zip(ks, inertias):
    print(f"k={k}: inertia={ine:>12,.0f}   (drop vs previous:"
          f" {-ine + inertias[k - 2] if k > 1 else float('nan'):,.0f})")

The drop from k=2→3 is dramatic; k=3→4 barely dents it. Elbows are judgement,
not law — combine this plot with domain sense ("marketing can only run 3–4
campaigns anyway").

## 5. Where K-Means Gets It Wrong

K-Means assumes clusters are roughly **spherical**, **similar-sized**, and defined by
**distance to a centre**. Feed it long snake-like or wildly unequal groups and it will
cheerfully carve them into nonsense wedges.

In [ ]:
# One elongated blob: K-Means slices it like a loaf instead of hugging its shape
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

rng = np.random.default_rng(7)
t = rng.uniform(0, 10, 300)
banana = np.column_stack([t, 0.35 * rng.normal(size=300)])   # long + thin

km_bad = KMeans(n_clusters=3, n_init=10, random_state=42).fit(banana)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

plt.figure(figsize=(6.5, 3.4))
for k in range(3):
    pts = banana[km_bad.labels_ == k]
    plt.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.75, color=colors[k])
plt.title("One natural group, sliced into 3 artificial ones")
plt.xlabel("x"); plt.ylabel("y"); plt.grid(alpha=0.3)
plt.show()

print("(DBSCAN or Gaussian mixtures handle shapes like this far better)")

## 6. PCA: Fewer Columns, Same Story

High-dimensional data is hard to plot, slow to train on, and mostly redundant —
neighbouring columns tend to say similar things (**curse of dimensionality**: as
dimensions grow, data thins out exponentially and distances lose meaning).

PCA rotates the axes to line up with the directions of greatest variance. The first
component captures as much spread as possible, the second captures the most of what's
left (perpendicular to the first), and so on. Keep the top two and you hold the
dataset's essence in plottable form.

In [ ]:
# Iris: 4 measurements per flower -> 2 principal components
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

X_scaled = StandardScaler().fit_transform(X)   # ALWAYS scale before PCA (next section!)
pca = PCA(n_components=2).fit(X_scaled)
X_2d = pca.transform(X_scaled)

print("components (rows = PC1, PC2):\n", pd.DataFrame(
    pca.components_, columns=X.columns, index=["PC1", "PC2"]).round(3))
print("\nvariance kept:", pca.explained_variance_ratio_.round(3),
      "-> total", round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
# How much story does each component tell?
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
import pandas as pd

X = pd.DataFrame(load_iris().data, columns=load_iris().feature_names)
ratios = PCA(n_components=4).fit(StandardScaler().fit_transform(X)).explained_variance_ratio_

plt.figure(figsize=(6.5, 3.6))
bars = plt.bar([f"PC{i+1}" for i in range(4)], ratios, color="#1f77b4", width=0.62)
bars[0].set_color("#ff7f0e"); bars[1].set_color("#ff7f0e")
plt.axhline(ratios.sum(), color="#7f7f7f", linestyle="--", linewidth=1)
plt.text(3.05, ratios.sum() + 0.01, f"all four = {ratios.sum():.2f}",
         ha="right", fontsize=9)
plt.title("Iris: two components carry ~96% of the variance")
plt.xlabel("principal component"); plt.ylabel("share of variance explained")
plt.ylim(0, 1.05); plt.grid(alpha=0.3, axis="y")
plt.show()

In [ ]:
# The payoff: a 4-D dataset drawn on ordinary paper, coloured by TRUE species
import matplotlib.pyplot as plt

species_names = ["setosa", "versicolor", "virginica"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

plt.figure(figsize=(6.5, 4))
for sp, name, c in zip(range(3), species_names, colors):
    plt.scatter(X_2d[y == sp, 0], X_2d[y == sp, 1], s=26, alpha=0.8,
                color=c, label=name)
plt.title("Iris in 2-D (PC1 vs PC2), colours = true species")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
plt.legend(); plt.grid(alpha=0.3)
plt.show()
print("Setosa separates completely; the other two overlap slightly.")

## 7. Scaling Before PCA — and PCA Before Modelling

Two habits to burn in:

- **Standardise first.** PCA maximises *variance*, and variance depends on units.
  Leave income in Taka and age in years, and PC1 simply becomes "income". Scaling puts
  all columns on one ruler so rotation measures signal, not units.
- **PCA as a pre-processing step.** Downstream models train faster on 2–20 components
  than on hundreds of raw columns, with almost no accuracy loss — noise directions get
  discarded along the way.

In [ ]:
# Speed + fidelity check on a wider synthetic problem
import time
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
n, d = 1500, 40
X_wide = rng.normal(size=(n, d))
y_wide = (X_wide[:, 0] - X_wide[:, 1] + rng.normal(0, 0.5, n) > 0).astype(int)

scaler = StandardScaler().fit(X_wide)
pca40 = PCA(n_components=0.95).fit(scaler.transform(X_wide))   # keep 95% variance
n_kept = pca40.n_components_
print(f"{d} raw features compressed to {n_kept} PCs keeping 95% of variance")

knn = KNeighborsClassifier(n_neighbors=5)

t0 = time.perf_counter()
acc_full = cross_val_score(knn, scaler.transform(X_wide), y_wide, cv=5).mean()
t_full = time.perf_counter() - t0

t0 = time.perf_counter()
acc_pca = cross_val_score(knn, pca40.transform(scaler.transform(X_wide)), y_wide, cv=5).mean()
t_pca = time.perf_counter() - t0

print(f"accuracy on {d} features : {acc_full:.3f}")
print(f"accuracy on {n_kept} PCs     : {acc_pca:.3f}")
print("(same idea, fewer columns - big speed win once d reaches hundreds/thousands)")

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Skipping scaling before PCA/K-Means | Big-unit columns dominate everything | `StandardScaler` inside the pipeline first |
| Picking k without evidence | Arbitrary segments nobody trusts | Elbow/silhouette plots plus domain reasoning |
| Expecting K-Means to find odd shapes | It carves snakes into wedges | DBSCAN / agglomerative / GMM instead |
| Fitting PCA on ALL data before splitting | Test variance leaks into components | Fit PCA on train only (Pipeline handles it) |
| Reading PCA components as features | Components are mixes of ALL original columns | Use `components_` loadings to interpret themes |
| Believing cluster IDs have inherent meaning | Labels 0/1/2 are arbitrary; rerun and they may swap | Interpret via centres/profiles, never by ID |

## 💡 Best Practices & Pro Tips

- **Profile every cluster** after fitting: means per feature per cluster turn group 2
  into *"high-spend weekend shoppers"* — that sentence is the deliverable.
- **Set `n_init=10` (or higher)** explicitly so results don't hinge on one lucky start.
- **Try `PCA(n_components=0.95)`** — keep enough components for 95% of variance and let
  sklearn choose the count for you.
- **Validate clusters for stability:** refit on bootstrap samples; groups that survive
  are real, groups that evaporate were noise.
- **AI-engineering relevance:** embeddings are exactly this pattern at scale —
  high-dimensional vectors reduced, clustered, and visualised with the very same
  tools (PCA/UMap + K-Means) used to debug LLM training corpora.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `make_blobs` / manual `rng.normal` | Seeded synthetic clusters | Ground truth known |
| `KMeans(n_clusters=k, n_init=10)` | Distance-to-centre clustering | `.fit_predict(X)` → labels |
| `kmeans.cluster_centers_` / `.inertia_` | Group centres / compactness score | Elbow method input |
| `PCA(n_components=2)` | Rotate onto max-variance axes | `.fit_transform(X_scaled)` |
| `pca.explained_variance_ratio_` | Variance share per component | Pick how many to keep |
| `StandardScaler` before either | Puts columns on one ruler | Mandatory, not optional |

Key takeaways:
- Without labels, success = interpretable, stable groups — not an accuracy number.
- K-Means is fast and beloved, but only for roundish, comparable-size blobs.
- The elbow in inertia is a heuristic compass, not a verdict.
- Scale before PCA or K-Means, always; PCA then feeds faster, leaner downstream models.

## 🔗 Next Lesson

Continue to **[07_Model_Evaluation_Pipelines](../07_Model_Evaluation_Pipelines/notes.ipynb)** —
the capstone: overfitting made visible, hyperparameter search, and pipelines that tie
this entire module together.